In [2]:
import numpy as np

In [3]:
delivery_data = np.genfromtxt(
            'deliveries.csv',
            delimiter= ',',
            dtype=None, 
            names = True,
            missing_values = '',
            filling_values = np.nan, 
            encoding='utf-8'
        )

In [4]:
delivery_data

array([( 335982, 1, 'Kolkata Knight Riders', 'Royal Challengers Bangalore',  0, 1, 'SC Ganguly', 'P Kumar', 'BB McCullum', 0, 1, 1, 'legbyes', 0, 'NA', 'NA', 'NA'),
       ( 335982, 1, 'Kolkata Knight Riders', 'Royal Challengers Bangalore',  0, 2, 'BB McCullum', 'P Kumar', 'SC Ganguly', 0, 0, 0, '', 0, 'NA', 'NA', 'NA'),
       ( 335982, 1, 'Kolkata Knight Riders', 'Royal Challengers Bangalore',  0, 3, 'BB McCullum', 'P Kumar', 'SC Ganguly', 0, 1, 1, 'wides', 0, 'NA', 'NA', 'NA'),
       ...,
       (1426312, 2, 'Kolkata Knight Riders', 'Sunrisers Hyderabad', 10, 1, 'VR Iyer', 'Shahbaz Ahmed', 'SS Iyer', 1, 0, 1, '', 0, 'NA', 'NA', 'NA'),
       (1426312, 2, 'Kolkata Knight Riders', 'Sunrisers Hyderabad', 10, 2, 'SS Iyer', 'Shahbaz Ahmed', 'VR Iyer', 1, 0, 1, '', 0, 'NA', 'NA', 'NA'),
       (1426312, 2, 'Kolkata Knight Riders', 'Sunrisers Hyderabad', 10, 3, 'VR Iyer', 'Shahbaz Ahmed', 'SS Iyer', 1, 0, 1, '', 0, 'NA', 'NA', 'NA')],
      shape=(260920,), dtype=[('match_id', '<i8'), ('i

In [5]:
delivery_data.dtype.names

('match_id',
 'inning',
 'batting_team',
 'bowling_team',
 'over',
 'ball',
 'batter',
 'bowler',
 'non_striker',
 'batsman_runs',
 'extra_runs',
 'total_runs',
 'extras_type',
 'is_wicket',
 'player_dismissed',
 'dismissal_kind',
 'fielder')

In [6]:
match_id = delivery_data['match_id'][:10]
match_id

array([335982, 335982, 335982, 335982, 335982, 335982, 335982, 335982,
       335982, 335982])

In [7]:
batting_team = delivery_data['batting_team'][:10]
batting_team

array(['Kolkata Knight Riders', 'Kolkata Knight Riders',
       'Kolkata Knight Riders', 'Kolkata Knight Riders',
       'Kolkata Knight Riders', 'Kolkata Knight Riders',
       'Kolkata Knight Riders', 'Kolkata Knight Riders',
       'Kolkata Knight Riders', 'Kolkata Knight Riders'], dtype='<U27')

In [8]:
batter = delivery_data['batter'][:10]
batter

array(['SC Ganguly', 'BB McCullum', 'BB McCullum', 'BB McCullum',
       'BB McCullum', 'BB McCullum', 'BB McCullum', 'BB McCullum',
       'BB McCullum', 'BB McCullum'], dtype='<U23')

In [9]:
bowler = delivery_data['bowler'][:10]
bowler

array(['P Kumar', 'P Kumar', 'P Kumar', 'P Kumar', 'P Kumar', 'P Kumar',
       'P Kumar', 'Z Khan', 'Z Khan', 'Z Khan'], dtype='<U23')

In [10]:
batsman_runs = delivery_data['batsman_runs'][:10]
batsman_runs

array([0, 0, 0, 0, 0, 0, 0, 0, 4, 4])

In [11]:
over = delivery_data['over'][:10]
over

array([0, 0, 0, 0, 0, 0, 0, 1, 1, 1])

### Total runs scored in each match

In [12]:
match_id = delivery_data['match_id']
batsman_runs = delivery_data['batsman_runs']

In [13]:
unique_matches = np.unique(match_id)

In [14]:
total_runs = []

for m in unique_matches:
    total_runs.append(batsman_runs[match_id == m].sum())

total_runs = np.array(total_runs)

print(total_runs)

# match_id_shifted = match_id - match_id.min()

#total_runs = np.bincount(match_id_shifted, weights=batsman_runs)

[268 430 244 ... 336 301 203]


In [15]:
result = np.column_stack((unique_matches, total_runs))
print(result)

[[ 335982     268]
 [ 335983     430]
 [ 335984     244]
 ...
 [1426310     336]
 [1426311     301]
 [1426312     203]]


### Top 5 batters based on total runs

In [16]:
batsman = delivery_data['batter']
batsman_runs = delivery_data['batsman_runs']

In [17]:
# Getting unique batters
unique_batters = np.unique(batsman)

In [18]:
# Computing total runs for each batter
total_runs = []

for b in unique_batters:
    runs = batsman_runs[batsman == b].sum()
    total_runs.append(runs)

total_runs = np.array(total_runs)

In [19]:
# indices of top 5 batters
top5_idx = np.argsort(total_runs)[-5:][::-1]

# top 5 batters and their runs
top5_batters = unique_batters[top5_idx]
top5_runs = total_runs[top5_idx]

# Display result
for i in range(5):
    print(top5_batters[i], ":", int(top5_runs[i]))

V Kohli : 8014
S Dhawan : 6769
RG Sharma : 6630
DA Warner : 6567
SK Raina : 5536


In [20]:
# Different Approach

# Sort by batsman
sorted_idx = np.argsort(batsman)
batsman_sorted = batsman[sorted_idx]
runs_sorted = batsman_runs[sorted_idx]

# unique batters and their start indices
unique_batters, indices = np.unique(batsman_sorted, return_index=True)

# Sum runs using split
total_runs = np.add.reduceat(runs_sorted, indices)

# Top 5
top5_idx = np.argsort(total_runs)[-5:][::-1]

for i in top5_idx:
    print(unique_batters[i], ":", int(total_runs[i]))

V Kohli : 8014
S Dhawan : 6769
RG Sharma : 6630
DA Warner : 6567
SK Raina : 5536


### Computing strike rate

In [22]:
batsman = delivery_data['batter']
batsman_runs = delivery_data['batsman_runs']

In [23]:
# Sort by batsman (required for grouping)
sorted_idx = np.argsort(batsman)

batsman_sorted = batsman[sorted_idx]
runs_sorted = batsman_runs[sorted_idx]

In [24]:
# Unique batters + group indices
unique_batters, indices = np.unique(batsman_sorted, return_index=True)

In [25]:
# Total runs per batter
total_runs = np.add.reduceat(runs_sorted, indices)

In [26]:
# Balls faced = count of deliveries
balls_faced = np.diff(np.append(indices, len(batsman_sorted)))

In [29]:
# Strike rate
strike_rate = (total_runs / balls_faced) * 100

In [31]:
for i in range(20):
    print(unique_batters[i], round(strike_rate[i], 2))

A Ashish Reddy 142.86
A Badoni 125.54
A Chandila 57.14
A Chopra 70.67
A Choudhary 125.0
A Dananjaya 80.0
A Flintoff 108.77
A Kamboj 100.0
A Kumble 71.43
A Manohar 127.62
A Mishra 86.59
A Mithun 130.77
A Mukund 82.61
A Nehra 65.08
A Nortje 96.08
A Raghuvanshi 149.54
A Singh 20.0
A Symonds 124.71
A Tomar 50.0
A Uniyal 57.14


### Economy Rate of Bowlers

In [32]:
bowler = delivery_data['bowler']
total_runs = delivery_data['total_runs']

In [33]:
# Sort by bowler
sorted_idx = np.argsort(bowler)

In [34]:
bowler_sorted = bowler[sorted_idx]
runs_sorted = total_runs[sorted_idx]

# Unique bowlers + indices
unique_bowlers, indices = np.unique(bowler_sorted, return_index=True)

In [35]:
# Total runs conceded
runs_conceded = np.add.reduceat(runs_sorted, indices)

# Balls bowled (count rows)
balls_bowled = np.diff(np.append(indices, len(bowler_sorted)))

In [36]:
# Economy rate
economy = (runs_conceded / balls_bowled) * 6

In [37]:
for i in range(20):
    print(unique_bowlers[i], round(economy[i], 2))

A Ashish Reddy 8.89
A Badoni 8.88
A Chandila 6.28
A Choudhary 8.0
A Dananjaya 11.28
A Flintoff 9.64
A Kamboj 10.15
A Kumble 6.65
A Mishra 7.3
A Mithun 9.17
A Nehra 7.71
A Nel 10.33
A Nortje 8.83
A Singh 7.89
A Symonds 7.71
A Uniyal 10.58
A Zampa 7.93
AA Chavan 7.98
AA Jhunjhunwala 8.86
AA Kazi 9.69


### Average runs per over

In [38]:
over = delivery_data['over']
batsman_runs = delivery_data['batsman_runs']

In [39]:
# Sort by over
sorted_idx = np.argsort(over)

over_sorted = over[sorted_idx]
runs_sorted = batsman_runs[sorted_idx]

In [40]:
# Unique overs + indices
unique_overs, indices = np.unique(over_sorted, return_index=True)

In [41]:
# Total runs per over
total_runs = np.add.reduceat(runs_sorted, indices)

# Balls per over
balls = np.diff(np.append(indices, len(over_sorted)))

In [42]:
# Average runs per ball in each over
avg_runs = total_runs / balls

In [44]:
for i in range(len(unique_overs)):
    print("Over", unique_overs[i] + 1, ":", round(avg_runs[i], 2))

Over 1 : 0.89
Over 2 : 1.08
Over 3 : 1.25
Over 4 : 1.29
Over 5 : 1.31
Over 6 : 1.31
Over 7 : 1.04
Over 8 : 1.14
Over 9 : 1.19
Over 10 : 1.17
Over 11 : 1.21
Over 12 : 1.24
Over 13 : 1.24
Over 14 : 1.29
Over 15 : 1.33
Over 16 : 1.37
Over 17 : 1.42
Over 18 : 1.5
Over 19 : 1.56
Over 20 : 1.67


### Total number of 4s and 6s

In [45]:
batsman_runs = delivery_data['batsman_runs']

# Count 4s
total_4s = np.sum(batsman_runs == 4)

# Count 6s
total_6s = np.sum(batsman_runs == 6)

print("Total 4s:", total_4s)
print("Total 6s:", total_6s)

Total 4s: 29850
Total 6s: 13051


#### Team with most boundaries

In [46]:
batting_team = delivery_data['batting_team']
batsman_runs = delivery_data['batsman_runs']

In [47]:
# Mask for boundaries
total_boundary = (batsman_runs == 4) | (batsman_runs == 6)

# Filter only boundary rows
teams_boundary = batting_team[total_boundary]

In [48]:
# Sort for grouping
sorted_idx = np.argsort(teams_boundary)
teams_sorted = teams_boundary[sorted_idx]

In [49]:
# Count boundaries per team
unique_teams, indices = np.unique(teams_sorted, return_index=True)
boundary_counts = np.diff(np.append(indices, len(teams_sorted)))

In [50]:
# Find team with max boundaries
top_idx = np.argmax(boundary_counts)

print("Team with most boundaries:", unique_teams[top_idx])
print("Total boundaries:", boundary_counts[top_idx])

Team with most boundaries: Mumbai Indians
Total boundaries: 5322


### Death over Analysis

In [51]:
over = delivery_data['over']
batsman_runs = delivery_data['batsman_runs']
batting_team = delivery_data['batting_team']

# Mask for death overs
death_over = (over >= 16) & (over <= 20)

In [52]:
total_death_runs = np.sum(batsman_runs[death_over])
print("Total runs in death overs:", total_death_runs)

Total runs in death overs: 71303


#### Team with highest runs in death overs

In [53]:
# Filter data for death overs
teams_death = batting_team[death_over]
runs_death = batsman_runs[death_over]

In [54]:
# Sort by team
sorted_idx = np.argsort(teams_death)

In [55]:
teams_sorted = teams_death[sorted_idx]
runs_sorted = runs_death[sorted_idx]

# Group by team
unique_teams, indices = np.unique(teams_sorted, return_index=True)

# Total runs per team
team_runs = np.add.reduceat(runs_sorted, indices)

In [56]:
# Find top team
top_idx = np.argmax(team_runs)

print("Top team in death overs:", unique_teams[top_idx])
print("Runs scored:", int(team_runs[top_idx]))

Top team in death overs: Mumbai Indians
Runs scored: 9069


### Highest Scoring match

In [63]:
match_id = delivery_data['match_id']
total_runs = delivery_data['total_runs']
batting_team = delivery_data['batting_team']
bowling_team = delivery_data['bowling_team']

In [59]:
# Sort by match_id
sorted_idx = np.argsort(match_id)

match_sorted = match_id[sorted_idx]
runs_sorted = total_runs[sorted_idx]

In [60]:
# Unique matches + indices
unique_matches, indices = np.unique(match_sorted, return_index=True)

# Total runs per match
match_runs = np.add.reduceat(runs_sorted, indices)

batting

In [64]:
# Find match with highest runs
top_idx = np.argmax(match_runs)

top_match_id = unique_matches[top_idx]


In [71]:
batting_team_in_match = np.unique(batting_team[match_id == top_match_id])[0]
bowling_team_in_match = np.unique(bowling_team[match_id == top_match_id])[1]


In [72]:
print("Match Id with highest runs:", unique_matches[top_idx])
print(batting_team_in_match, " vs " + bowling_team_in_match)
print("Total runs scored:", int(match_runs[top_idx]))

Match Id with highest runs: 1426268
Royal Challengers Bengaluru  vs Sunrisers Hyderabad
Total runs scored: 549


### Runs per team per Match

In [73]:
match_id = delivery_data['match_id']
batting_team = delivery_data['batting_team']
total_runs = delivery_data['total_runs']

# Combining match_id + team
combined = match_id.astype(str) + "_" + batting_team

In [74]:
# Sorting by combined key
sorted_idx = np.argsort(combined)

combined_sorted = combined[sorted_idx]
runs_sorted = total_runs[sorted_idx]

In [75]:
# Unique match-team pairs
unique_keys, indices = np.unique(combined_sorted, return_index=True)

# Total runs per (match, team)
runs_per_team_match = np.add.reduceat(runs_sorted, indices)

In [76]:
# Extracting match_id and team
match_ids = np.array([key.split("_")[0] for key in unique_keys])
teams = np.array([key.split("_")[1] for key in unique_keys])

In [ ]:
for i in range(5):
    print("Match:", match_ids[i],
          "| Team:", teams[i],
          "| Runs:", int(runs_per_team_match[i]))

Match: 1082591 | Team: Royal Challengers Bangalore | Runs: 172
Match: 1082591 | Team: Sunrisers Hyderabad | Runs: 207
Match: 1082592 | Team: Mumbai Indians | Runs: 184
Match: 1082592 | Team: Rising Pune Supergiant | Runs: 187
Match: 1082593 | Team: Gujarat Lions | Runs: 183


### Match winner Approximation

In [82]:
match_id = delivery_data['match_id']
batting_team = delivery_data['batting_team']
total_runs = delivery_data['total_runs']

combined = match_id.astype(str) + "_" + batting_team

sorted_idx = np.argsort(combined)

combined_sorted = combined[sorted_idx]
runs_sorted = total_runs[sorted_idx]

unique_keys, indices = np.unique(combined_sorted, return_index=True)
runs_per_team = np.add.reduceat(runs_sorted, indices)

# Extract match_id + team
match_ids = np.array([key.split("_")[0] for key in unique_keys])
teams = np.array([key.split("_")[1] for key in unique_keys])

In [83]:
# Sort by match_id
sorted_idx = np.argsort(match_ids)

match_ids_sorted = match_ids[sorted_idx]
teams_sorted = teams[sorted_idx]
runs_sorted = runs_per_team[sorted_idx]

unique_matches, indices = np.unique(match_ids_sorted, return_index=True)

In [84]:
winners = []

for i in range(len(indices)):
    start = indices[i]
    end = indices[i+1] if i+1 < len(indices) else len(match_ids_sorted)
    
    match_teams = teams_sorted[start:end]
    match_runs = runs_sorted[start:end]
    
    winner = match_teams[np.argmax(match_runs)]
    winners.append(winner)

winners = np.array(winners)

In [85]:
for i in range(5):
    print("Match:", unique_matches[i], "| Winner:", winners[i])

Match: 1082591 | Winner: Sunrisers Hyderabad
Match: 1082592 | Winner: Rising Pune Supergiant
Match: 1082593 | Winner: Kolkata Knight Riders
Match: 1082594 | Winner: Kings XI Punjab
Match: 1082595 | Winner: Royal Challengers Bangalore


### Score Card

In [86]:
match_id = delivery_data['match_id']
batting_team = delivery_data['batting_team']
total_runs = delivery_data['total_runs']

# Combine keys
combined = match_id.astype(str) + "_" + batting_team

# Sort
sorted_idx = np.argsort(combined)

combined_sorted = combined[sorted_idx]
runs_sorted = total_runs[sorted_idx]

# Group
unique_keys, indices = np.unique(combined_sorted, return_index=True)
runs_per_team = np.add.reduceat(runs_sorted, indices)

# Extract match_id + team
match_ids = np.array([key.split("_")[0] for key in unique_keys])
teams = np.array([key.split("_")[1] for key in unique_keys])

In [87]:
# Sort by match_id
sorted_idx = np.argsort(match_ids)

match_ids_sorted = match_ids[sorted_idx]
teams_sorted = teams[sorted_idx]
runs_sorted = runs_per_team[sorted_idx]

# Get match boundaries
unique_matches, indices = np.unique(match_ids_sorted, return_index=True)

In [92]:
for i in range(10):
    start = indices[i]
    end = indices[i+1] if i+1 < len(indices) else len(match_ids_sorted)
    
    print(f"Match {unique_matches[i]}:")
    
    for j in range(start, end):
        print(f" {teams_sorted[j]}: {int(runs_sorted[j])} runs")
    
    print()

Match 1082591:
 Royal Challengers Bangalore: 172 runs
 Sunrisers Hyderabad: 207 runs

Match 1082592:
 Mumbai Indians: 184 runs
 Rising Pune Supergiant: 187 runs

Match 1082593:
 Gujarat Lions: 183 runs
 Kolkata Knight Riders: 184 runs

Match 1082594:
 Kings XI Punjab: 164 runs
 Rising Pune Supergiant: 163 runs

Match 1082595:
 Delhi Daredevils: 142 runs
 Royal Challengers Bangalore: 157 runs

Match 1082596:
 Gujarat Lions: 135 runs
 Sunrisers Hyderabad: 140 runs

Match 1082597:
 Kolkata Knight Riders: 178 runs
 Mumbai Indians: 180 runs

Match 1082598:
 Royal Challengers Bangalore: 148 runs
 Kings XI Punjab: 150 runs

Match 1082599:
 Delhi Daredevils: 205 runs
 Rising Pune Supergiant: 108 runs

Match 1082600:
 Mumbai Indians: 159 runs
 Sunrisers Hyderabad: 158 runs

